<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:OrderLine</h2>

----

### Вариант задания 15 


<h2 style="color:DodgerBlue">Описание проекта:</h2>


----

Создать базовый класс OrderLine в C#, который будет представлять информацию о 
строке заказа, содержащей детали одного товара в заказе. На основе этого класса 
разработать 2-3 производных класса, демонстрирующих принципы наследования и 
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и 
методы,  а  также  переопределены  некоторые  методы  базового  класса  для 
демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [7]:
using System;
using System.Collections.Generic;

interface IDiscountable
{
    decimal Discount { get; set; }
    decimal GetDiscountAmount();
}


interface IShippable
{
    string Address { get; set; }
    decimal DeliveryCost { get; set; }
    decimal GetTotalWithDelivery();
}


class OrderLine
{
    public int ProductId { get; set; }
    public string ProductName { get; set; }
    public decimal Price { get; set; }
    public string Category { get; set; } = "Общая";
    public DateTime CreatedAt { get; set; } = DateTime.Now;
    public bool IsActive { get; set; } = true;
    public decimal Weight { get; set; } = 1;

    public OrderLine(int productId, string productName, decimal price)
    {
        ProductId = productId;
        ProductName = productName;
        Price = price;
    }

    public virtual decimal CalculateTotal()
    {
        return Price;
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        Price = newPrice;
    }

    public virtual string GetProductDetails()
    {
        return $"ID товара: {ProductId}, Название: {ProductName}, Цена: {Price} руб.";
    }

    public void Activate()
    {
        IsActive = true;
    }

    public void Deactivate()
    {
        IsActive = false;
    }

    public string GetStatus()
    {
        return $"Товар {ProductName}: категория {Category}, вес {Weight} кг, " +
               (IsActive ? "активен" : "неактивен");
    }

    public bool IsExpensive(decimal limit)
    {
        return Price > limit;
    }
}


class StandardLine : OrderLine
{
    public int Units { get; set; }
    public string Warehouse { get; set; } = "Основной склад";
    public bool IsReserved { get; set; } = false;
    public int MinUnits { get; set; } = 1;
    public string Supplier { get; set; } = "Не указан";

    public StandardLine(int productId, string productName, decimal price, int units)
        : base(productId, productName, price)
    {
        Units = units;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Количество: {Units} шт.";
    }

    public void IncreaseUnits(int count)
    {
        Units += count;
    }

    public void DecreaseUnits(int count)
    {
        Units = Math.Max(MinUnits, Units - count);
    }

    public void Reserve()
    {
        IsReserved = true;
    }

    public string GetWarehouseInfo()
    {
        return $"Склад: {Warehouse}, поставщик: {Supplier}, " +
               (IsReserved ? "в резерве" : "без резерва");
    }
}


class SpecialLine : OrderLine
{
    public decimal Discount { get; set; }
    public string PromoCode { get; set; } = "NONE";
    public DateTime PromoEnd { get; set; } = DateTime.Now.AddDays(30);
    public bool IsPromoActive { get; set; } = true;
    public decimal MaxDiscount { get; set; } = 50;

    public SpecialLine(int productId, string productName, decimal price, decimal discount)
        : base(productId, productName, price)
    {
        Discount = discount;
    }

    public override void UpdatePrice(decimal newPrice)
    {
        Price = newPrice - newPrice * Discount / 100;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Скидка: {Discount}%";
    }

    public void ExtendPromo(int days)
    {
        PromoEnd = PromoEnd.AddDays(days);
    }

    public void ClosePromo()
    {
        IsPromoActive = false;
    }

    public decimal GetDiscountAmount(decimal basePrice)
    {
        return basePrice * Math.Min(Discount, MaxDiscount) / 100;
    }

    public string GetPromoInfo()
    {
        return $"Промокод: {PromoCode}, действует до {PromoEnd:dd.MM.yyyy}, " +
               (IsPromoActive ? "активен" : "закрыт");
    }
}


class FreeLine : OrderLine
{
    public decimal Prepayment { get; set; }
    public bool IsPrepaid { get; set; } = false;
    public DateTime? PaymentDate { get; set; } = null;
    public string PaymentMethod { get; set; } = "Не выбран";

    public FreeLine(int productId, string productName, decimal price, decimal prepayment)
        : base(productId, productName, price)
    {
        Prepayment = prepayment;
    }

    public override decimal CalculateTotal()
    {
        return Price - Prepayment;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Предварительный платеж: {Prepayment} руб.";
    }

    public void PayPrepayment(string method)
    {
        PaymentMethod = method;
        PaymentDate = DateTime.Now;
        IsPrepaid = true;
    }

    public decimal GetRemainder()
    {
        return Price - Prepayment;
    }

    public bool IsFullyPaid()
    {
        return Prepayment >= Price;
    }

    public string GetPaymentInfo()
    {
        return IsPrepaid
            ? $"Предоплата внесена {PaymentDate:dd.MM.yyyy}, способ: {PaymentMethod}"
            : "Предоплата не внесена";
    }
}


class PremiumLine : StandardLine
{
    public int WarrantyMonths { get; set; }
    public string SupportLevel { get; set; } = "Стандартная";
    public bool IsGiftWrapped { get; set; } = false;

    public PremiumLine(int productId, string productName, decimal price, int units, int warrantyMonths)
        : base(productId, productName, price, units)
    {
        WarrantyMonths = warrantyMonths;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Гарантия: {WarrantyMonths} мес.";
    }

    public void ExtendWarranty(int months)
    {
        WarrantyMonths += months;
    }

    public void WrapAsGift()
    {
        IsGiftWrapped = true;
    }

    public string GetWarrantyInfo()
    {
        return $"Гарантия {WarrantyMonths} мес., поддержка: {SupportLevel}, " +
               (IsGiftWrapped ? "подарочная упаковка" : "обычная упаковка");
    }
}


class CombinedLine : OrderLine, IDiscountable, IShippable
{
    public int Units { get; set; }
    public decimal Discount { get; set; }
    public string Address { get; set; }
    public decimal DeliveryCost { get; set; }

    public CombinedLine(int productId, string productName, decimal price, int units,
                        decimal discount, string address, decimal deliveryCost)
        : base(productId, productName, price)
    {
        Units = units;
        Discount = discount;
        Address = address;
        DeliveryCost = deliveryCost;
    }

    public decimal GetDiscountAmount()
    {
        return Price * Units * Discount / 100;
    }

    public decimal GetTotalWithDelivery()
    {
        return CalculateTotal() + DeliveryCost;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units - GetDiscountAmount();
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Количество: {Units} шт., Скидка: {Discount}%, Адрес: {Address}";
    }
}


class Order
{
    private int _orderId;
    private Customer _customer;
    private List<OrderLine> _lines;

    public Order(int orderId, Customer customer)
    {
        _orderId = orderId;
        _customer = customer;
        _lines = new List<OrderLine>();
    }

    public int OrderId
    {
        get { return _orderId; }
        set { _orderId = value; }
    }

    public Customer Customer
    {
        get { return _customer; }
        set { _customer = value; }
    }

    public List<OrderLine> Lines
    {
        get { return _lines; }
        set { _lines = value; }
    }

    public void AddLine(OrderLine line)
    {
        _lines.Add(line);
    }

    public decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (OrderLine line in _lines)
        {
            total += line.CalculateTotal();
        }
        return total;
    }

    public string GetOrderDetails()
    {
        string result = $"Заказ №{OrderId}, клиент: {Customer.Name}";
        foreach (OrderLine line in _lines)
        {
            result += Environment.NewLine + "  " + line.GetProductDetails();
        }
        return result;
    }
}


class Customer
{
    private int _customerId;
    private string _name;
    private List<Order> _orders;

    public Customer(int customerId, string name)
    {
        _customerId = customerId;
        _name = name;
        _orders = new List<Order>();
    }

    public int CustomerId
    {
        get { return _customerId; }
        set { _customerId = value; }
    }

    public string Name
    {
        get { return _name; }
        set { _name = value; }
    }

    public List<Order> Orders
    {
        get { return _orders; }
        set { _orders = value; }
    }

    public void AddOrder(Order order)
    {
        order.Customer = this;
        _orders.Add(order);
    }

    public decimal GetTotalSpent()
    {
        decimal total = 0;
        foreach (Order order in _orders)
        {
            total += order.CalculateTotal();
        }
        return total;
    }
}


OrderLine standard = new StandardLine(
    1, "Ноутбук", 50000, 2);

OrderLine special = new SpecialLine(
    2, "Телефон", 30000, 10);

OrderLine free = new FreeLine(
    3, "Наушники", 5000, 1000);


Console.WriteLine("СТАНДАРТНАЯ СТРОКА");
Console.WriteLine(standard.GetProductDetails());
Console.WriteLine("Общая стоимость: " +
                  standard.CalculateTotal() + " руб.");

Console.WriteLine();


Console.WriteLine("СПЕЦИАЛЬНАЯ СТРОКА");
Console.WriteLine(special.GetProductDetails());

special.UpdatePrice(30000);

Console.WriteLine("Цена со скидкой: " +
                  special.Price + " руб.");

Console.WriteLine();


Console.WriteLine("БЕСПЛАТНАЯ СТРОКА");
Console.WriteLine(free.GetProductDetails());
Console.WriteLine("Общая стоимость после вычета предоплаты: " +
                  free.CalculateTotal() + " руб.");

Console.WriteLine();

Customer customer = new Customer(1, "Иван Петров");
Order order = new Order(100, customer);
order.AddLine(standard);
order.AddLine(special);
order.AddLine(free);
customer.AddOrder(order);

Console.WriteLine("ЗАКАЗ");
Console.WriteLine(order.GetOrderDetails());
Console.WriteLine("Стоимость заказа: " +
                  order.CalculateTotal() + " руб.");
Console.WriteLine("Всего потрачено клиентом " + customer.Name + ": " +
                  customer.GetTotalSpent() + " руб.");

Console.WriteLine();

Console.WriteLine("ПРОСТОЕ НАСЛЕДОВАНИЕ");
StandardLine standardLine = (StandardLine)standard;
standardLine.Warehouse = "Склад №2";
standardLine.Supplier = "ТехноОпт";
standardLine.Reserve();
standardLine.IncreaseUnits(3);
Console.WriteLine(standardLine.GetStatus());
Console.WriteLine(standardLine.GetWarehouseInfo());
Console.WriteLine("Количество: " + standardLine.Units +
                  ", стоимость: " + standardLine.CalculateTotal() + " руб.");

SpecialLine specialLine = (SpecialLine)special;
specialLine.PromoCode = "AUTUMN10";
specialLine.ExtendPromo(15);
Console.WriteLine(specialLine.GetPromoInfo());
Console.WriteLine("Размер скидки от 30000: " +
                  specialLine.GetDiscountAmount(30000) + " руб.");

FreeLine freeLine = (FreeLine)free;
Console.WriteLine(freeLine.GetPaymentInfo());
freeLine.PayPrepayment("Карта");
Console.WriteLine(freeLine.GetPaymentInfo());
Console.WriteLine("Остаток к оплате: " + freeLine.GetRemainder() + " руб.");

Console.WriteLine();

Console.WriteLine("МНОГОУРОВНЕВОЕ НАСЛЕДОВАНИЕ");
PremiumLine premium = new PremiumLine(
    4, "Планшет", 40000, 1, 12);
premium.ExtendWarranty(6);
premium.WrapAsGift();
premium.SupportLevel = "VIP";
premium.Category = "Электроника";
Console.WriteLine(premium.GetProductDetails());
Console.WriteLine(premium.GetWarrantyInfo());
Console.WriteLine(premium.GetStatus());
Console.WriteLine("Общая стоимость: " + premium.CalculateTotal() + " руб.");

Console.WriteLine();

Console.WriteLine("МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ");
CombinedLine combined = new CombinedLine(
    5, "Монитор", 15000, 2, 10, "г. Екатеринбург, ул. Ленина, 1", 500);
Console.WriteLine(combined.GetProductDetails());
Console.WriteLine("Скидка: " + combined.GetDiscountAmount() + " руб.");
Console.WriteLine("Стоимость: " + combined.CalculateTotal() + " руб.");
Console.WriteLine("Стоимость с доставкой: " +
                  combined.GetTotalWithDelivery() + " руб.");

IDiscountable discountable = combined;
IShippable shippable = combined;
Console.WriteLine("Через IDiscountable: " + discountable.GetDiscountAmount() + " руб.");
Console.WriteLine("Через IShippable: " + shippable.Address);

order.AddLine(premium);
order.AddLine(combined);
Console.WriteLine();
Console.WriteLine("Стоимость заказа после добавления новых строк: " +
                  order.CalculateTotal() + " руб.");

СТАНДАРТНАЯ СТРОКА
ID товара: 1, Название: Ноутбук, Цена: 50000 руб., Количество: 2 шт.
Общая стоимость: 100000 руб.

СПЕЦИАЛЬНАЯ СТРОКА
ID товара: 2, Название: Телефон, Цена: 30000 руб., Скидка: 10%
Цена со скидкой: 27000 руб.

БЕСПЛАТНАЯ СТРОКА
ID товара: 3, Название: Наушники, Цена: 5000 руб., Предварительный платеж: 1000 руб.
Общая стоимость после вычета предоплаты: 4000 руб.

ЗАКАЗ
Заказ №100, клиент: Иван Петров
  ID товара: 1, Название: Ноутбук, Цена: 50000 руб., Количество: 2 шт.
  ID товара: 2, Название: Телефон, Цена: 27000 руб., Скидка: 10%
  ID товара: 3, Название: Наушники, Цена: 5000 руб., Предварительный платеж: 1000 руб.
Стоимость заказа: 131000 руб.
Всего потрачено клиентом Иван Петров: 131000 руб.

ПРОСТОЕ НАСЛЕДОВАНИЕ
Товар Ноутбук: категория Общая, вес 1 кг, активен
Склад: Склад №2, поставщик: ТехноОпт, в резерве
Количество: 5, стоимость: 250000 руб.
Промокод: AUTUMN10, действует до 02.12.2026, активен
Размер скидки от 30000: 3000 руб.
Предоплата не внесена
Предопла